# Demo 2 — Agent + Knowledge (File Search)

Ground a Microsoft Agent Framework agent in a research-protocol document using a Foundry-hosted **vector store** and the `HostedFileSearchTool`.

**Prereqs**
- `pip install -r requirements.txt`
- A populated `.env`
- `az login`

## 1. Load environment

In [ ]:
import os, glob
from dotenv import load_dotenv

load_dotenv()

PROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

DATA_FILES = glob.glob("data/*")
print("Files to upload:", DATA_FILES)

## 2. Upload documents and build a vector store

We use the `azure-ai-projects` SDK to talk to the Foundry project's Agents endpoint and create a vector store from our local files.

In [ ]:
from azure.ai.projects.aio import AIProjectClient
from azure.ai.agents.models import FilePurpose
from azure.identity.aio import AzureCliCredential

credential = AzureCliCredential()
projects = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

uploaded_file_ids = []
for path in DATA_FILES:
    f = await projects.agents.files.upload_and_poll(file_path=path, purpose=FilePurpose.AGENTS)
    uploaded_file_ids.append(f.id)
    print(f"Uploaded {path} -> {f.id}")

vector_store = await projects.agents.vector_stores.create_and_poll(
    file_ids=uploaded_file_ids,
    name="ped-all-protocol-vs",
)
print("Vector store id:", vector_store.id)

## 3. Create the agent with `HostedFileSearchTool`

In [ ]:
from agent_framework import HostedFileSearchTool, HostedVectorStoreContent
from agent_framework.azure import AzureAIAgentClient

client = AzureAIAgentClient(
    project_endpoint=PROJECT_ENDPOINT,
    model_deployment_name=MODEL_DEPLOYMENT,
    async_credential=credential,
)

file_search = HostedFileSearchTool(
    inputs=[HostedVectorStoreContent(vector_store_id=vector_store.id)]
)

agent = client.create_agent(
    name="ProtocolQAAgent",
    instructions=(
        "You are a research-operations assistant. Answer questions about the attached "
        "study protocol using ONLY the provided documents. Quote or paraphrase the "
        "protocol text. If the answer is not in the documents, say so explicitly and "
        "recommend the user consult the PI or study coordinator."
    ),
    tools=[file_search],
)

## 4. Ask grounded questions

In [ ]:
print(await agent.run("What is the primary objective of PED-ALL-2025-01, and what's the primary endpoint window?"))

In [ ]:
print(await agent.run("Summarize the inclusion and exclusion criteria. I need to screen a candidate."))

In [ ]:
print(await agent.run("At which time points do we collect bone marrow versus cfDNA samples?"))

In [ ]:
# Out-of-scope sanity check — the agent should refuse rather than confabulate.
print(await agent.run("What is the recommended starting dose of EX-1042 for adults with AML?"))

## 5. Cleanup

Remove the demo vector store and files so they don't linger in the Foundry project.

In [ ]:
await projects.agents.vector_stores.delete(vector_store.id)
for fid in uploaded_file_ids:
    await projects.agents.files.delete(fid)
await client.close()
await projects.close()
await credential.close()
print("Cleaned up.")